# Task 5 — density + autoencoder experiment

The supplied `density_auto` CSV gives predictions only, not its implementation. This notebook tests a specified unsupervised hypothesis: diagonal GMM low-density rank plus shallow autoencoder reconstruction-error rank, weighted equally. The user confirmed the lecturer permits fitting on unlabeled Task 5 `test.csv`. The CSV labels are used only for descriptive comparison, never for fitting or choosing model settings. No offline F3 can be computed.

In [1]:
from pathlib import Path
import sys, json, joblib
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/pipeline/save_model.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
SRC = ROOT / "src"
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.task5.density_auto import DensityAutoDetector
SEED = set_seed()
DATA_NAME = "eda_original_v2"
card = json.loads((SRC / "task5/data/eda" / f"{DATA_NAME}.json").read_text(encoding="utf-8"))
assert set(card["splits"]) == {"test"}
frame = pd.read_parquet(SRC / card["splits"]["test"]["path"])
assert len(frame) == 25889 and frame.player_id.is_unique and "is_anomaly" not in frame
X = frame.drop(columns="player_id")
assert X.shape[1] == 122 and all(pd.api.types.is_numeric_dtype(t) for t in X.dtypes)
print("Test-only data", X.shape, "missing cells", int(X.isna().sum().sum()))

Test-only data (25889, 122) missing cells 154627


## Unlabeled stability protocol

Use a 12,000-account fit, an overlapping 10,000-account alternate fit, and the same separate 4,000-account holdout used in earlier Task 5 detector comparisons. Top-8% Jaccard measures rank repeatability only.

In [2]:
rows = np.arange(len(X))
dev_idx, hold_idx = train_test_split(rows, train_size=12000, test_size=4000, random_state=SEED)
alt_idx = np.random.default_rng(SEED).choice(dev_idx, size=10000, replace=False)
assert not set(dev_idx).intersection(hold_idx)
def top_jaccard(a, b, fraction=0.08):
    k = round(len(a) * fraction)
    top_a = set(np.lexsort((np.arange(len(a)), -np.asarray(a)))[:k])
    top_b = set(np.lexsort((np.arange(len(b)), -np.asarray(b)))[:k])
    return len(top_a & top_b) / len(top_a | top_b)

## Fit, save, and evaluate one named model

In [3]:
NAME = "t5_density_auto_original_v1"
paths = [SRC / "task5/save" / f"{NAME}{suffix}" for suffix in (".joblib", ".json", "-evaluation.json")]
present = [p.exists() for p in paths]
if any(present) and not all(present):
    raise RuntimeError(f"Incomplete artifact for {NAME}; inspect before rerunning")
if all(present):
    model = joblib.load(paths[0])
    model_card = json.loads(paths[1].read_text(encoding="utf-8"))
    evaluation = json.loads(paths[2].read_text(encoding="utf-8"))
    assert model_card["data_name"] == DATA_NAME and evaluation["metric"] == "top8_rank_stability_jaccard"
else:
    model = DensityAutoDetector(anomaly_fraction=0.08, random_state=SEED, max_iter=40).fit(X.iloc[dev_idx])
    alternate = DensityAutoDetector(anomaly_fraction=0.08, random_state=SEED, max_iter=40).fit(X.iloc[alt_idx])
    stability = top_jaccard(model.anomaly_score(X.iloc[hold_idx]), alternate.anomaly_score(X.iloc[hold_idx]))
    save_model(task=5, name=NAME, model=model, data_name=DATA_NAME,
               card={"idea": "Equal percentile-rank blend of PCA-space diagonal GMM rarity and MLP autoencoder reconstruction error",
                     "features": "122 original numeric features after existing coordinate mask",
                     "preprocessing": "median imputation; fitted normal-quantile transform clipped to [-4,4]",
                     "architecture": "MLP 122-64-16-64-122; 4-component diagonal GMM on 12 PCA scores",
                     "fit_rows": 12000, "seed": SEED, "scope": "lecturer-permitted unsupervised fit on Task 5 test-only file"})
    evaluation = evaluate_model(task=5, name=NAME, score=stability,
        metric_name="top8_rank_stability_jaccard", higher_is_better=True,
        card={"meaning": "Top-8% Jaccard of two overlapping fits scored on the same 4,000-account holdout; not F3",
              "alternate_fit_rows": 10000, "seed": SEED})
print(NAME, "stability proxy", evaluation["score"], "MLP iterations", model.autoencoder_.n_iter_)

C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


t5_density_auto_original_v1 stability proxy 0.6537467700258398 MLP iterations 40


## Describe scores and compare predictions

The friend's CSV is a reference prediction, not ground truth. Equal alert counts allow a set comparison without fitting or threshold tuning to its labels.

In [4]:
FRIEND = Path.home() / "Downloads" / "kaggle_t12345_t1_slow800_bagging_v4_t3_conditional_quantile_v1_t4_resnet18_cosine18_tta_v1_t5_density_auto_v1.csv"
friend = pd.read_csv(FRIEND)
sample = pd.read_csv(SRC / "submission/sample_submission.csv", usecols=["id"])
assert len(friend) == len(frame) == len(sample) and friend.id.equals(sample.id)
assert set(friend.task5.unique()) == {0, 1}
scores = model.anomaly_score(X)
assert np.isfinite(scores).all()
k = int(friend.task5.sum())  # comparison budget only; never used in fitting
selected = np.lexsort((np.arange(len(scores)), -scores))[:k]
pred = np.zeros(len(scores), dtype=np.int8); pred[selected] = 1
reference = friend.task5.to_numpy(dtype=np.int8)
comparison = pd.crosstab(pd.Series(reference, name="friend"), pd.Series(pred, name="ours"))
display(comparison)
print("friend alerts", k, "our alerts", int(pred.sum()), "matching positives", int(((pred == 1) & (reference == 1)).sum()))
print("Jaccard vs friend predictions", round(int(((pred == 1) & (reference == 1)).sum()) / int(((pred == 1) | (reference == 1)).sum()), 5))
print("This is prediction agreement, not accuracy or F3.")

ours,0,1
friend,,
0,22375,1468
1,1468,578


friend alerts 2046 our alerts 2046 matching positives 578
Jaccard vs friend predictions 0.16448
This is prediction agreement, not accuracy or F3.
